In [ ]:
import pyBigWig
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



base_url = "/path_to_your_bigwig_dir"
 


# BigWigs
bw_tt_dmso = pyBigWig.open(f"{base_url}/DMSO_Mean.bw")
bw_tt_trt  = pyBigWig.open(f"{base_url}/ISD_Mean.bw")
bw_pol_dmso = pyBigWig.open(f"{base_url}/DMSO_PolI_merged.bw")
bw_pol_trt  = pyBigWig.open(f"{base_url}/ISD_PolI_merged.bw")

# Regions: chrom start end name
bed = pd.read_csv(
    "chrR.bed",
    sep="\t",
    header=None,
    usecols=[0, 1, 2, 3],
    names=["chrom", "start", "end", "name"]

)


## Check if there is monotonic coupling of Pol I occupancy with TT-seq response

In [ ]:
def region_mean(bw, chrom, start, end):
    try:
        vals = bw.values(chrom, int(start), int(end), numpy=True)
        return np.nanmean(vals) if vals is not None else 0.0
    except RuntimeError:
        return 0.0



records = []

for _, r in bed.iterrows():
    tt_d = region_mean(bw_tt_dmso, r.chrom, r.start, r.end)
    tt_t = region_mean(bw_tt_trt,  r.chrom, r.start, r.end)
    pol_d = region_mean(bw_pol_dmso, r.chrom, r.start, r.end)
    pol_t = region_mean(bw_pol_trt,  r.chrom, r.start, r.end)

    records.append({
        "region": r.name,
        "tt_dmso": tt_d,
        "tt_trt": tt_t,
        "pol_dmso": pol_d,
        "pol_trt": pol_t
    })

df = pd.DataFrame(records)



PSEUDO = 1e-9

df["dTT"]  = np.log2((df.tt_trt  + PSEUDO) / (df.tt_dmso  + PSEUDO))
df["dPol"] = np.log2((df.pol_trt + PSEUDO) / (df.pol_dmso + PSEUDO))


df = df.sort_values("dTT").reset_index(drop=True)

df["decile"] = pd.qcut(df["dTT"], 10, labels=False)


summary = (
    df.groupby("decile")
      .agg(
          mean_dPol=("dPol", "mean"),
          sem_dPol=("dPol", lambda x: np.std(x, ddof=1) / np.sqrt(len(x))),
          n=("dPol", "size")
      )
      .reset_index()
)



plt.figure(figsize=(6,4))

plt.errorbar(
    summary.decile,
    summary.mean_dPol,
    yerr=summary.sem_dPol,
    fmt='o-',
    capsize=3
)

plt.xlabel("ΔTT-seq decile (low → high)")
plt.ylabel("Mean ΔPol I (log2 FC)")
plt.title("Monotonic coupling of Pol I occupancy with TT-seq response")

plt.axhline(0, color="black", lw=0.8, ls="--")
plt.tight_layout()
plt.show()

## Check if there is monotonic coupling of , TT-seq response and Pol I coupling

In [ ]:
df = df.sort_values("dPol").reset_index(drop=True)
df["decile"] = pd.qcut(df["dPol"], 10, labels=False)

summary = (
    df.groupby("decile")
      .agg(
          mean_dTT=("dTT", "mean"),
          sem_dTT=("dTT", lambda x: np.std(x, ddof=1) / np.sqrt(len(x))),
          n=("dTT", "size")
      )
      .reset_index()
)
plt.figure(figsize=(6,4))

plt.errorbar(
    summary.decile,
    summary.mean_dTT,
    yerr=summary.sem_dTT,
    fmt='o-',
    capsize=3
)

plt.xlabel("ΔPol I decile (low → high)")
plt.ylabel("Mean ΔTT-seq (log2 FC)")
plt.title("TT-seq response as a function of Pol I occupancy")

plt.axhline(0, color="black", lw=0.8, ls="--")
plt.tight_layout()
plt.show()


## Compute nucleotide lag in Pol I occupancy and TT-seq response

In [ ]:
import pyBigWig
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

def get_lag_and_correlation(chip_bw, tt_bw, chrom, start, end):
    # Open BigWigs
    bw1 = pyBigWig.open(chip_bw)
    bw2 = pyBigWig.open(tt_bw)
    
    # Get signal as numpy arrays (handle NaNs)
    # interval is the resolution (e.g., 10bp bins)
    chip_sig = np.nan_to_num(np.array(bw1.values(chrom, start, end)))
    tt_sig = np.nan_to_num(np.array(bw2.values(chrom, start, end)))
    
    # Normalize signals (Z-score) to make them comparable
    chip_sig = (chip_sig - np.mean(chip_sig)) / np.std(chip_sig)
    tt_sig = (tt_sig - np.mean(tt_sig)) / np.std(tt_sig)
    
    # Calculate Cross-Correlation
    correlation = signal.correlate(tt_sig, chip_sig, mode='full')
    lags = signal.correlation_lags(len(tt_sig), len(chip_sig), mode='full')
    
    # Find the lag with the Maximum Correlation
    max_corr_idx = np.argmax(correlation)
    optimal_lag = lags[max_corr_idx]
    max_corr_val = correlation[max_corr_idx]
    
    return optimal_lag, max_corr_val, lags, correlation

# Define your rDNA coordinates
chrom = "chrR"
start = 1
end = 44838 # End of rDNA unit

# Calculate for DMSO
lag_dmso, corr_dmso, lags_axis, corr_curve_dmso = get_lag_and_correlation(
    f"{base_url}/DMSO_PolI_merged.bw", f"{base_url}/DMSO_Mean.bw", chrom, start, end
)

# Calculate for Treatment
lag_treat, corr_treat, _, corr_curve_treat = get_lag_and_correlation(
    f"{base_url}/ISD_PolI_merged.bw", f"{base_url}/ISD_Mean.bw", chrom, start, end
)

print(f"DMSO Lag: {lag_dmso} bp")
print(f"Treatment Lag: {lag_treat} bp")

# Plotting the Cross-Correlation Curve
plt.figure(figsize=(10, 5))
plt.plot(lags_axis, corr_curve_dmso, label=f'DMSO (Lag: {lag_dmso}bp)', color='blue')
plt.plot(lags_axis, corr_curve_treat, label=f'Treatment (Lag: {lag_treat}bp)', color='red')
plt.title(f"Pol I vs TT-seq Signal Cross-Correlation ({chrom})")
plt.xlabel("Lag (bp)")
plt.ylabel("Cross-Correlation Score")
plt.legend()
plt.xlim(-500, 2000) # Zoom in on the relevant lag window
plt.grid(True, alpha=0.3)
#plt.savefig("rDNA_Lag_Analysis.pdf")

# Pol I efficiency evaluation


In [ ]:
import pyBigWig
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Define your regions of interest (Coding sequences where elongation happens)
# Replace these coordinates with your actual rDNA annotations


# Add coords
regions = {
    "18S": ("chrR", 12996, 14864)
    "28S": ("chrR", 17259, 22309),
    "5'_ETS": ("chrR", 9339, 12995) 
}

# 2. Function to get mean signal from BigWig
def get_mean_signal(bw_object, chrom, start, end):
    # type='mean' gets average signal over the region
    return bw_object.stats(chrom, start, end, type="mean")[0]

# 3. Extract data
data = []

for region_name, (chrom, start, end) in regions.items():
    # Get DMSO signal
    dmso_val = get_mean_signal(DMSO_eff, chrom, start, end)
    data.append({"Condition": "DMSO", "Region": region_name, "Efficiency": dmso_val})
    
    # Get ISD signal
    isd_val = get_mean_signal(ISD_eff, chrom, start, end)
    data.append({"Condition": "ISD", "Region": region_name, "Efficiency": isd_val})

df = pd.DataFrame(data)

# 4. Plotting
plt.figure(figsize=(6, 5))
sns.barplot(data=df, x="Region", y="Efficiency", hue="Condition", palette=["navy", "firebrick"])
plt.title("Pol I Elongation Efficiency (TT-seq / ChIP)")
plt.ylabel("Elongation Efficiency Score")
plt.show()